# MyCityAI — train the complaint-photo model (YOLOv8) on Google Colab

**Before you start**
1. On your PC: `python ml/vision/build_dataset.py --name civic-v0 --zip` → creates `ml/vision/data/civic-v0.zip`.
2. Upload `civic-v0.zip` to your Google Drive in a folder called **`MyCityAI`**.
3. Here in Colab: **Runtime → Change runtime type → T4 GPU**, then run the cells from top to bottom.

Training saves to Google Drive after every epoch, so if Colab disconnects just run all cells again — it continues where it stopped.
See `docs/ML.md` for class names, datasets and how to deploy the result.

In [ ]:
# 1. Check the GPU (must show a Tesla T4 or similar)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Settings — change DATASET when you build a newer version (civic-v1, ...)
DATASET = "civic-v0"
RUN_NAME = "v0"
MODEL = "yolov8s.pt"   # small model: good speed / accuracy balance
EPOCHS = 100
IMGSZ = 640
BATCH = 16

DRIVE_DIR = "/content/drive/MyDrive/MyCityAI"
RUNS_DIR = f"{DRIVE_DIR}/runs"          # results are kept on Drive

In [ ]:
# 3. Connect Google Drive and unpack the dataset
from google.colab import drive
drive.mount("/content/drive")

import os, zipfile
zip_path = f"{DRIVE_DIR}/{DATASET}.zip"
data_dir = f"/content/{DATASET}"
assert os.path.exists(zip_path), f"Upload {DATASET}.zip to Google Drive → MyCityAI/ first"
if not os.path.exists(data_dir):
    zipfile.ZipFile(zip_path).extractall(data_dir)
print(open(f"{data_dir}/stats.json").read()[:1500])

In [ ]:
# 4. Install Ultralytics YOLOv8
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

In [ ]:
# 5. Train (resumes automatically if a previous run on Drive was interrupted)
from pathlib import Path
from ultralytics import YOLO

last = Path(RUNS_DIR) / RUN_NAME / "weights" / "last.pt"
if last.exists():
    print("Resuming", last)
    model = YOLO(str(last))
    model.train(resume=True)
else:
    model = YOLO(MODEL)  # starts from COCO-pretrained weights (transfer learning)
    model.train(
        data=f"{data_dir}/data.yaml",
        epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
        patience=20,                  # stop early if validation stops improving
        project=RUNS_DIR, name=RUN_NAME, exist_ok=True,
        fliplr=0.5, flipud=0.0,       # no upside-down roads
        hsv_v=0.25, degrees=10,       # brightness and small rotations (docs/ML.md §3)
    )

In [ ]:
# 6. Evaluate on the test split (never seen during training) — numbers for the project report
best = YOLO(f"{RUNS_DIR}/{RUN_NAME}/weights/best.pt")
metrics = best.val(data=f"{data_dir}/data.yaml", split="test", project=RUNS_DIR, name=f"{RUN_NAME}-test", exist_ok=True)
print(f"Test mAP@0.5      = {metrics.box.map50:.3f}")
print(f"Test mAP@0.5:0.95 = {metrics.box.map:.3f}")
print(f"Precision = {metrics.box.mp:.3f}   Recall = {metrics.box.mr:.3f}")
print()
for i, name in best.names.items():
    if i in metrics.box.ap_class_index:
        k = list(metrics.box.ap_class_index).index(i)
        print(f"  {name:20s} mAP@0.5 = {metrics.box.ap50[k]:.3f}   mAP@0.5:0.95 = {metrics.box.ap[k]:.3f}")
    else:
        print(f"  {name:20s} (no test images yet)")

In [ ]:
# 7. Save the model for the backend: renamed copy + checksum on Google Drive
import hashlib, shutil
final = f"{DRIVE_DIR}/mycityai-yolov8s-{RUN_NAME}.pt"
shutil.copy(f"{RUNS_DIR}/{RUN_NAME}/weights/best.pt", final)
digest = hashlib.sha256(open(final, "rb").read()).hexdigest()
print("Model:", final)
print("sha256:", digest)
print()
print("Next (docs/ML.md §5): copy the .pt to municipal/backend/models/, add it to ml/vision/models.json")
print("with this sha256, and record the test numbers above in ml/README.md.")

In [ ]:
# 8. Try it on a few test photos
import glob
from IPython.display import Image, display
samples = sorted(glob.glob(f"{data_dir}/test/images/*"))[:6]
for result in best.predict(samples, conf=0.4, save=True, project=RUNS_DIR, name=f"{RUN_NAME}-samples", exist_ok=True):
    display(Image(filename=f"{result.save_dir}/{Path(result.path).name}", width=480))